# Download UCF-Crime directly from Kaggle

This downloads `odins0n/ucf-crime-dataset` directly from Kaggle to the Colab runtime. It does not use your home upload bandwidth.

Select **Runtime → Change runtime type → T4 GPU** before running the later training notebook.

In [ ]:
# Mount Drive so the project code and training results persist.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
%cd /content/drive/MyDrive/Course Project
PROJECT_DIR = Path.cwd()
print('Project:', PROJECT_DIR)

In [ ]:
# Install KaggleHub. Public datasets normally download without manually uploading an API key.
!pip -q install -U kagglehub

In [ ]:
# Download from Kaggle into Colab's local cache.
# It may be large and can take a while, but it uses Colab's connection rather than your PC's upload speed.
import kagglehub

KAGGLE_DATASET = 'odins0n/ucf-crime-dataset'
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print('Downloaded dataset path:', dataset_path)

In [ ]:
# Inspect the first two levels before training. This prevents using the wrong Kaggle layout by accident.
for entry in sorted(dataset_path.iterdir(), key=lambda item: item.name.lower()):
    print(entry.name + ('/' if entry.is_dir() else ''))
    if entry.is_dir():
        for child in sorted(entry.iterdir(), key=lambda item: item.name.lower())[:12]:
            print('   └─ ' + child.name + ('/' if child.is_dir() else ''))

In [ ]:
# Find possible ready-to-use roots containing both Train/ and Test/.
candidates = []
for folder in [dataset_path, *[path for path in dataset_path.rglob('*') if path.is_dir()]]:
    if (folder / 'Train').is_dir() and (folder / 'Test').is_dir():
        candidates.append(folder)

if candidates:
    print('Ready-to-use dataset root(s):')
    for candidate in candidates:
        print(' ', candidate)
else:
    print('No Train/Test folder pair was found. Stop here and share the output of the previous cell.')
    print('This Kaggle version may contain raw videos and need a conversion/split step before it fits this project.')

## Create lists and train from the Kaggle cache

The dataset remains on Colab's fast local disk. We do not copy it into Google Drive or create a Drive symlink. The small list files, model checkpoint, metrics, and ROC image are saved in the project folder on Drive.

In [ ]:
# Create data/subset list files using the ready-to-use Kaggle dataset root.
import subprocess
subprocess.run(
    ['python', 'scripts/setup_environment.py', '--data-root', str(dataset_path)],
    check=True,
)

In [ ]:
# Install dependencies and verify that the T4 can run the R3D-18 model.
!bash scripts/setup_colab.sh

In [ ]:
# Train directly from Colab's local Kaggle cache.
# Lower --batch-size from 8 to 4 if Colab reports CUDA out of memory.
subprocess.run(
    ['python', 'scripts/train_supervised.py',
     '--data-root', str(dataset_path),
     '--train-list', 'data/subset/ucf_crime_train_subset.txt',
     '--val-list', 'data/subset/ucf_crime_test_subset.txt',
     '--epochs', '30', '--patience', '5', '--batch-size', '8', '--num-workers', '2'],
    check=True,
)